# 03. Entrenamiento con MobileNetV2
Este notebook realiza el entrenamiento del modelo MobileNetV2 siguiendo el mismo flujo de dos fases.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

sys.path.append('..')
from utils import preprocessing, augmentation, metrics

RANDOM_STATE = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

## 1. Carga de Datos y Generadores

In [ ]:
df = preprocessing.load_data_to_df('../data/')
df_imb = preprocessing.create_artificial_imbalance(df, random_state=RANDOM_STATE)
train_df, val_df, test_df = preprocessing.split_data(df_imb, random_state=RANDOM_STATE)

train_gen_obj, val_gen_obj = augmentation.get_augmentation_generator(model_type='mobilenet')

train_generator = train_gen_obj.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_generator = val_gen_obj.flow_from_dataframe(
    val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_generator = val_gen_obj.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

## 2. FASE 1: Transfer Learning Puro

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(23, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('../models/mobilenet_phase1.keras', save_best_only=True)
]

history_phase1 = model.fit(train_generator, validation_data=val_generator, epochs=10, callbacks=callbacks)
metrics.plot_history(history_phase1, title="MobileNetV2 - Fase 1")

## 3. FASE 2: Fine-Tuning

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_ft = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('../models/mobilenet_best.keras', save_best_only=True)
]

history_phase2 = model.fit(train_generator, validation_data=val_generator, epochs=10, callbacks=callbacks_ft)
metrics.plot_history(history_phase2, title="MobileNetV2 - Fase 2")
metrics.compare_phases(history_phase1, history_phase2)

## 4. Evaluación y Experimentos

In [ ]:
y_true = test_generator.classes
y_pred = np.argmax(model.predict(test_generator), axis=1)
classes = list(test_generator.class_indices.keys())

metrics.plot_confusion_matrix(y_true, y_pred, classes)
metrics.print_classification_report(y_true, y_pred, classes)
metrics.analyze_overfitting(history_phase2)

np.save('../results/mobilenet_history.npy', history_phase2.history)